# 07 TumorBoard Qualitative Evaluation

这个 notebook 是 **06 主消融实验的补充**，不重跑 150 道 MCQ。

目标：从自建 Tumor Board 病例中抽 5 个复杂病例，对比：

- **E1**：单 Agent 强 API baseline
- **E4**：ClawTeam 完整 MDT（外科/内科 LoRA + 病理/放疗 API + Round 2 + Coordinator 仲裁）

然后用 LLM-as-judge 按 4 个维度评分：

1. `medical_accuracy`
2. `plan_completeness`
3. `mdt_reasoning`
4. `actionable_timeline`

注意：这是一组 qualitative / auxiliary evaluation，不是医学金标准。

## Step 0: 独立环境准备

In [1]:
import os
import sys
import json
import random
import time
from pathlib import Path

import pandas as pd

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None


def find_backend_dir() -> Path:
    """Find backend directory whether the notebook is run locally or on AutoDL."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for p in candidates:
        if (p / "app.py").exists() and (p / "graph").exists():
            return p
        if (p / "backend" / "app.py").exists() and (p / "backend" / "graph").exists():
            return p / "backend"

    autodl = Path("/root/autodl-tmp/ClowTeam_NLP/backend")
    if autodl.exists():
        return autodl

    local = Path(r"D:\NLP_Project\miniOpenClaw\backend")
    if local.exists():
        return local

    raise RuntimeError("Cannot locate backend directory. Please run this notebook inside ClowTeam_NLP/backend or backend/eval/notebooks.")


BACKEND_DIR = find_backend_dir()
REPO_DIR = BACKEND_DIR.parent
DATA_DIR = BACKEND_DIR / "eval" / "datasets" / "data"
RESULTS_DIR = BACKEND_DIR / "eval" / "results" / "tumor_board_qualitative"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(BACKEND_DIR)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

if load_dotenv is not None:
    load_dotenv(BACKEND_DIR / "config" / ".env")

TUMOR_BOARD_QUAL_N = int(os.getenv("TUMOR_BOARD_QUAL_N", "5"))
TUMOR_BOARD_QUAL_SEED = int(os.getenv("TUMOR_BOARD_QUAL_SEED", "42"))
TUMOR_BOARD_PATH = DATA_DIR / "tumor_board_cases.jsonl"

QUAL_RESULTS_CSV = RESULTS_DIR / "tumor_board_qualitative_results.csv"
QUAL_JUDGE_JSONL = RESULTS_DIR / "tumor_board_qualitative_judge.jsonl"
QUAL_SUMMARY_CSV = RESULTS_DIR / "tumor_board_qualitative_summary.csv"

print("BACKEND_DIR =", BACKEND_DIR)
print("DATA_DIR =", DATA_DIR)
print("RESULTS_DIR =", RESULTS_DIR)
print("TUMOR_BOARD_PATH exists =", TUMOR_BOARD_PATH.exists())
print("TUMOR_BOARD_QUAL_N =", TUMOR_BOARD_QUAL_N)

BACKEND_DIR = /root/autodl-tmp/ClowTeam_NLP/backend
DATA_DIR = /root/autodl-tmp/ClowTeam_NLP/backend/eval/datasets/data
RESULTS_DIR = /root/autodl-tmp/ClowTeam_NLP/backend/eval/results/tumor_board_qualitative
TUMOR_BOARD_PATH exists = True
TUMOR_BOARD_QUAL_N = 5


## Step 1: 导入后端模块并配置 E4 LoRA

In [2]:
from config import get_settings
from graph.llm import build_llm_config_from_settings, get_llm
from graph.coordinator import Coordinator
from graph.complexity_assessor import CaseComplexity


DEFAULT_QWEN3_BASE = (
    os.getenv("QWEN3_4B_BASE")
    or os.getenv("LORA_SURGEON_BASE")
    or os.getenv("LORA_MEDICAL_ONCOLOGIST_BASE")
    or "/root/autodl-tmp/Qwen3-4B-Instruct-2507"
)


def configure_e4_lora_env() -> None:
    """Force E4 intended backend: surgeon/oncologist LoRA; pathology/radiation role APIs from .env."""
    os.environ["USE_LORA_SURGEON"] = "true"
    os.environ["USE_LORA_MEDICAL_ONCOLOGIST"] = "true"
    os.environ["USE_LOCAL_BASE_SURGEON"] = "false"
    os.environ["USE_LOCAL_BASE_MEDICAL_ONCOLOGIST"] = "false"

    os.environ.setdefault("LORA_SURGEON_BASE", DEFAULT_QWEN3_BASE)
    os.environ.setdefault("LORA_MEDICAL_ONCOLOGIST_BASE", DEFAULT_QWEN3_BASE)
    os.environ.setdefault("LORA_SURGEON_PATH", "eval/models/surgeon_qwen3_lora")
    os.environ.setdefault("LORA_MEDICAL_ONCOLOGIST_PATH", "eval/models/oncologist_qwen3_lora")

    # Clear local model caches if the module has already been imported in this kernel.
    try:
        import eval.inference.load_lora_role as lora_loader

        for cache_name in ["_lora_cache", "_local_base_cache"]:
            cache = getattr(lora_loader, cache_name, None)
            if isinstance(cache, dict):
                cache.clear()
    except Exception as exc:
        print("[warn] Could not clear LoRA caches:", repr(exc))


def get_global_api_llm(temperature: float = 0.2):
    settings = get_settings()
    return get_llm(build_llm_config_from_settings(settings, temperature=temperature, streaming=False))


print("Global LLM provider/model:", os.getenv("LLM_PROVIDER"), os.getenv("LLM_MODEL"))
print("Pathologist model:", os.getenv("PATHOLOGIST_LLM_MODEL"), "| vision:", os.getenv("PATHOLOGIST_VISION_MODEL"))
print("Radiation model:", os.getenv("RADIATION_ONCOLOGIST_LLM_MODEL"))
print("Qwen3 base:", DEFAULT_QWEN3_BASE)
print("Surgeon LoRA path:", os.getenv("LORA_SURGEON_PATH", "eval/models/surgeon_qwen3_lora"))
print("Oncologist LoRA path:", os.getenv("LORA_MEDICAL_ONCOLOGIST_PATH", "eval/models/oncologist_qwen3_lora"))

Global LLM provider/model: deepseek deepseek-chat
Pathologist model: qwen3.6-plus | vision: qwen3.6-plus
Radiation model: qwen3-max
Qwen3 base: /root/autodl-tmp/Qwen3-4B-Instruct-2507
Surgeon LoRA path: eval/models/surgeon_qwen3_lora
Oncologist LoRA path: eval/models/oncologist_qwen3_lora


## Step 2: 加载 TumorBoard 病例

In [3]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def load_tumor_board_for_qualitative(limit: int = 5) -> list[dict]:
    if not TUMOR_BOARD_PATH.exists():
        raise FileNotFoundError(f"TumorBoard cases not found: {TUMOR_BOARD_PATH}")

    selected_ids = [
        x.strip()
        for x in os.getenv("TUMOR_BOARD_CASE_IDS", "").split(",")
        if x.strip()
    ]

    cases = []
    for i, item in enumerate(read_jsonl(TUMOR_BOARD_PATH), 1):
        case_text = item.get("case") or item.get("question") or item.get("input") or ""
        case_text = str(case_text).strip()
        if not case_text:
            continue

        case = {
            "id": item.get("id", f"TumorBoard-{i}"),
            "title": item.get("title", ""),
            "tumor_type": item.get("tumor_type", ""),
            "complexity": item.get("complexity", ""),
            "source": "TumorBoard",
            "question": case_text,
            "question_stem": case_text,
            "options": [],
            "answer": item.get("reference_plan", item.get("expected_answer", item.get("answer", ""))),
            "raw": item,
        }
        cases.append(case)

    if selected_ids:
        by_id = {c["id"]: c for c in cases}
        chosen = [by_id[x] for x in selected_ids if x in by_id]
        if not chosen:
            raise ValueError(f"TUMOR_BOARD_CASE_IDS was set but no ids matched: {selected_ids}")
        return chosen[:limit]

    rng = random.Random(TUMOR_BOARD_QUAL_SEED)
    # Prefer complex and information-rich cases; deterministic shuffle within top pool.
    cases = sorted(
        cases,
        key=lambda c: (str(c.get("complexity", "")).lower() == "complex", len(c["question"])),
        reverse=True,
    )
    top_pool = cases[: min(len(cases), max(limit * 3, limit))]
    rng.shuffle(top_pool)
    return top_pool[:limit]


qual_cases = load_tumor_board_for_qualitative(TUMOR_BOARD_QUAL_N)
print(f"Loaded {len(qual_cases)} TumorBoard qualitative cases")
for c in qual_cases:
    print("-", c["id"], "|", c.get("title", ""), "| chars=", len(c["question"]))

Loaded 5 TumorBoard qualitative cases
- tb_008 | 三阴性乳腺癌 BRCA1 阳性 | chars= 97
- tb_025 | 肺腺癌 KRAS G12C 二线 | chars= 87
- tb_007 | 乳腺癌 HER2+ ER+ 局部进展期 | chars= 101
- tb_029 | 胃癌 Claudin18.2 阳性新靶点 | chars= 102
- tb_011 | 食管鳞癌 T2N1M0 治疗策略 | chars= 82


## Step 3: 定义 E1 / E4 运行函数

In [4]:
def build_tumor_board_prompt(case: dict) -> str:
    return (
        "你是肿瘤 MDT 会诊系统。请基于下面病例给出综合治疗方案。\n"
        "要求：\n"
        "1. 给出诊断/分期/关键分子信息；\n"
        "2. 给出治疗路径和时间线；\n"
        "3. 说明手术、系统治疗、放疗的角色；\n"
        "4. 给出条件分支和风险提醒；\n"
        "5. 不要编造病例中没有的信息。\n\n"
        f"病例：\n{case['question']}"
    )


async def run_qual_e1_single(case: dict) -> tuple[str, float]:
    """Single strong API baseline for a free-form TumorBoard case."""
    llm = get_global_api_llm(temperature=0.2)

    start = time.monotonic()
    response = await llm.ainvoke([
        {
            "role": "system",
            "content": (
                "You are a senior oncology tumor board assistant. "
                "Return a clinically structured treatment plan in Chinese. "
                "Do not fabricate missing imaging/pathology details."
            ),
        },
        {"role": "user", "content": build_tumor_board_prompt(case)},
    ])
    return getattr(response, "content", "").strip(), time.monotonic() - start


async def run_qual_e4_mdt(case: dict) -> tuple[str, object, float]:
    """Full ClawTeam MDT: LoRA surgeon/oncologist + API pathology/radiation + Round 2."""
    configure_e4_lora_env()
    coordinator = Coordinator(BACKEND_DIR)

    start = time.monotonic()
    session = await coordinator.consult(
        case=build_tumor_board_prompt(case),
        force_complexity=CaseComplexity.COMPLEX,
        skip_round2=False,
    )
    return session.final_decision, session, time.monotonic() - start

## Step 4: Judge 与审计工具

In [5]:
def collect_model_calls(session) -> list[dict]:
    if session is None:
        return []
    calls = []
    for attr in ["round1_opinions", "round2_opinions"]:
        for opinion in getattr(session, attr, []) or []:
            calls.extend(getattr(opinion, "tool_calls", []) or [])
    return calls


def summarize_model_calls(calls: list[dict]) -> dict:
    backends = [str(c.get("backend", "")) for c in calls]
    return {
        "model_calls": len(calls),
        "api_calls": sum(1 for b in backends if b.startswith("api")),
        "lora_calls": sum(1 for b in backends if b == "lora"),
        "local_base_calls": sum(1 for b in backends if b == "local_base"),
        "vision_api_calls": sum(1 for b in backends if "vision" in b.lower()),
        "backend_trace": "|".join(sorted(set(b for b in backends if b))),
    }


def extract_json_object(text: str) -> dict:
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except Exception:
            pass

    return {"parse_error": True, "raw": text}


async def judge_e1_vs_e4(case: dict, e1_output: str, e4_output: str) -> dict:
    """LLM judge for free-form MDT quality. A/B order is randomized per case."""
    judge_llm = get_global_api_llm(temperature=0.0)

    rng = random.Random(str(case.get("id", "")) + str(TUMOR_BOARD_QUAL_SEED))
    swap = rng.choice([False, True])
    if swap:
        system_a_name, system_a_text = "E4_ClawTeam", e4_output
        system_b_name, system_b_text = "E1_single", e1_output
    else:
        system_a_name, system_a_text = "E1_single", e1_output
        system_b_name, system_b_text = "E4_ClawTeam", e4_output

    reference = case.get("answer", "")
    reference_block = f"\n参考方案（若有，仅作辅助，不是唯一标准）：\n{reference}\n" if reference else ""

    judge_prompt = f"""
你是医学 NLP 评测裁判。请比较两个系统对同一个 Tumor Board 病例的回答质量。

病例：
{case['question']}

{reference_block}

系统 A 输出：
{system_a_text}

系统 B 输出：
{system_b_text}

请按以下 4 个维度分别给 A 和 B 打 1-5 分：
1. medical_accuracy：医学准确性与安全性
2. plan_completeness：治疗方案完整度
3. mdt_reasoning：多专科协作、分歧处理、仲裁体现
4. actionable_timeline：时间线、条件分支、可执行性

要求：
- 不要因为某个回答更长就自动给高分；
- 如果回答编造病例中不存在的信息，要扣分；
- 如果回答有明确多专科权衡、条件分支、风险提醒，可在 mdt_reasoning 和 actionable_timeline 加分；
- 只输出 JSON，不要输出 Markdown。

JSON 格式：
{{
  "A": {{
    "medical_accuracy": 1,
    "plan_completeness": 1,
    "mdt_reasoning": 1,
    "actionable_timeline": 1,
    "brief_reason": "..."
  }},
  "B": {{
    "medical_accuracy": 1,
    "plan_completeness": 1,
    "mdt_reasoning": 1,
    "actionable_timeline": 1,
    "brief_reason": "..."
  }},
  "winner": "A/B/tie",
  "overall_reason": "..."
}}
""".strip()

    response = await judge_llm.ainvoke([
        {
            "role": "system",
            "content": (
                "You are a strict medical NLP evaluator. "
                "Return valid JSON only. Do not add markdown."
            ),
        },
        {"role": "user", "content": judge_prompt},
    ])
    raw = getattr(response, "content", "").strip()
    parsed = extract_json_object(raw)
    parsed["_raw_judge_output"] = raw
    parsed["_system_a_name"] = system_a_name
    parsed["_system_b_name"] = system_b_name
    return parsed


def flatten_judge(case: dict, judge: dict, e1_latency: float, e4_latency: float, e4_session=None) -> dict:
    row = {
        "case_id": case.get("id"),
        "title": case.get("title", ""),
        "tumor_type": case.get("tumor_type", ""),
        "complexity": case.get("complexity", ""),
        "e1_latency_s": e1_latency,
        "e4_latency_s": e4_latency,
    }

    calls = collect_model_calls(e4_session)
    row.update({f"e4_{k}": v for k, v in summarize_model_calls(calls).items()})

    a_name = judge.get("_system_a_name")
    b_name = judge.get("_system_b_name")
    side_for = {a_name: "A", b_name: "B"}

    for method in ["E1_single", "E4_ClawTeam"]:
        side = side_for.get(method)
        score_obj = judge.get(side, {}) if side else {}
        prefix = "e1" if method == "E1_single" else "e4"

        for metric in ["medical_accuracy", "plan_completeness", "mdt_reasoning", "actionable_timeline"]:
            try:
                row[f"{prefix}_{metric}"] = float(score_obj.get(metric, 0))
            except Exception:
                row[f"{prefix}_{metric}"] = 0.0
        row[f"{prefix}_brief_reason"] = score_obj.get("brief_reason", "")

    for prefix in ["e1", "e4"]:
        vals = [
            row.get(f"{prefix}_medical_accuracy", 0),
            row.get(f"{prefix}_plan_completeness", 0),
            row.get(f"{prefix}_mdt_reasoning", 0),
            row.get(f"{prefix}_actionable_timeline", 0),
        ]
        row[f"{prefix}_avg_score"] = sum(vals) / len(vals)

    winner_side = judge.get("winner", "tie")
    if winner_side == side_for.get("E1_single"):
        winner = "E1_single"
    elif winner_side == side_for.get("E4_ClawTeam"):
        winner = "E4_ClawTeam"
    else:
        winner = "tie"

    row["winner"] = winner
    row["overall_reason"] = judge.get("overall_reason", "")
    row["judge_parse_error"] = bool(judge.get("parse_error", False))
    return row

## Step 5: 跑 5 个 TumorBoard 案例

In [6]:
qual_rows = []
qual_raw_records = []

for i, case in enumerate(qual_cases, 1):
    print(f"\n=== TumorBoard qualitative {i}/{len(qual_cases)}: {case['id']} ===")

    e1_output, e1_latency = await run_qual_e1_single(case)
    print(f"E1 done: {e1_latency:.1f}s, chars={len(e1_output)}")

    e4_output, e4_session, e4_latency = await run_qual_e4_mdt(case)
    print(f"E4 done: {e4_latency:.1f}s, chars={len(e4_output)}")

    judge = await judge_e1_vs_e4(case, e1_output, e4_output)
    print("Judge winner:", judge.get("winner"), "| A =", judge.get("_system_a_name"), "| B =", judge.get("_system_b_name"))

    flat = flatten_judge(case, judge, e1_latency, e4_latency, e4_session)
    qual_rows.append(flat)

    qual_raw_records.append({
        "case_id": case.get("id"),
        "title": case.get("title", ""),
        "case": case.get("question"),
        "reference": case.get("answer", ""),
        "e1_output": e1_output,
        "e4_output": e4_output,
        "judge": judge,
    })

qual_df = pd.DataFrame(qual_rows)
qual_df.to_csv(QUAL_RESULTS_CSV, index=False, encoding="utf-8-sig")

with QUAL_JUDGE_JSONL.open("w", encoding="utf-8") as f:
    for record in qual_raw_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("\nSaved CSV:", QUAL_RESULTS_CSV)
print("Saved raw JSONL:", QUAL_JUDGE_JSONL)
qual_df


=== TumorBoard qualitative 1/5: tb_008 ===
E1 done: 19.7s, chars=2199


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

E4 done: 196.4s, chars=2198
Judge winner: B | A = E1_single | B = E4_ClawTeam

=== TumorBoard qualitative 2/5: tb_025 ===
E1 done: 23.2s, chars=2874


...

E4 done: 237.3s, chars=2070
Judge winner: A | A = E4_ClawTeam | B = E1_single

=== TumorBoard qualitative 5/5: tb_011 ===
E1 done: 19.9s, chars=2241


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

E4 done: 194.0s, chars=2067
Judge winner: B | A = E1_single | B = E4_ClawTeam

Saved CSV: /root/autodl-tmp/ClowTeam_NLP/backend/eval/results/tumor_board_qualitative/tumor_board_qualitative_results.csv
Saved raw JSONL: /root/autodl-tmp/ClowTeam_NLP/backend/eval/results/tumor_board_qualitative/tumor_board_qualitative_judge.jsonl


,case_id,title,tumor_type,complexity,e1_latency_s,e4_latency_s,e4_model_calls,e4_api_calls,e4_lora_calls,e4_local_base_calls,...,e4_medical_accuracy,e4_plan_completeness,e4_mdt_reasoning,e4_actionable_timeline,e4_brief_reason,e1_avg_score,e4_avg_score,winner,overall_reason,judge_parse_error
0,tb_008,三阴性乳腺癌 BRCA1 阳性,breast,complex,19.693040,196.417680,10,4,4,0,...,5.0,5.0,5.0,5.0,方案准确，严格遵循OlympiA研究证据，明确反对新辅助同步使用奥拉帕利，并详细阐述了多专科...,4.25,5.00,E4_ClawTeam,系统B在医学准确性上更优，明确纠正了系统A中未明确禁止的新辅助同步使用奥拉帕利的潜在风险，且...,False
1,tb_025,肺腺癌 KRAS G12C 二线,lung,complex,23.184590,188.543128,10,4,4,0,...,5.0,5.0,5.0,5.0,高度准确，严格遵循指南，明确反对不合理联合用药，详细覆盖脑/骨转移局部处理、靶向单药、时序、...,3.25,5.00,E4_ClawTeam,系统A在医学准确性、方案完整度、MDT推理和可执行时间线四个维度均显著优于系统B。A不仅给出...,False
2,tb_007,乳腺癌 HER2+ ER+ 局部进展期,breast,complex,21.657592,191.691743,10,4,4,0,...,4.0,4.0,5.0,4.0,MDT推理出色，明确仲裁分歧（内分泌时长、放疗时序），但内分泌治疗时长建议（pCR 2年）偏...,4.75,4.25,E1_single,系统A在医学准确性、方案完整性和可执行性上更优，严格遵循指南（如内分泌治疗5年、T-DM1用...,False
3,tb_029,胃癌 Claudin18.2 阳性新靶点,gastric,complex,18.585083,237.265010,10,4,4,0,...,5.0,5.0,5.0,5.0,严格遵循循证医学，准确指出围手术期使用Zolbetuximab缺乏证据，推荐标准手术+辅助化...,2.25,5.00,E4_ClawTeam,系统A在医学准确性上明显优于系统B，严格遵循当前指南和证据，而系统B错误地将晚期一线证据外推...,False
4,tb_011,食管鳞癌 T2N1M0 治疗策略,esophageal,complex,19.909186,193.986742,10,4,4,0,...,5.0,5.0,5.0,5.0,医学准确、方案完整、时间线清晰，且明确展示了多专科分歧、仲裁依据和条件分支，MDT推理出色。,4.50,5.00,E4_ClawTeam,两个系统在医学准确性和方案完整性上均优秀，但系统B在MDT推理维度明显更优，明确展示了多专科...,False


## Step 6: 汇总表（poster/report 可用）

In [7]:
display_cols = [
    "case_id",
    "winner",
    "e1_avg_score",
    "e4_avg_score",
    "e1_medical_accuracy",
    "e4_medical_accuracy",
    "e1_plan_completeness",
    "e4_plan_completeness",
    "e1_mdt_reasoning",
    "e4_mdt_reasoning",
    "e1_actionable_timeline",
    "e4_actionable_timeline",
]

print("=== TumorBoard qualitative per-case summary ===")
print(qual_df[display_cols].to_string(index=False))

metric_rows = []
for metric in ["medical_accuracy", "plan_completeness", "mdt_reasoning", "actionable_timeline", "avg_score"]:
    e1 = qual_df[f"e1_{metric}"].mean()
    e4 = qual_df[f"e4_{metric}"].mean()
    metric_rows.append({
        "metric": metric,
        "E1_single": e1,
        "E4_ClawTeam": e4,
        "delta_E4_minus_E1": e4 - e1,
    })

qual_summary = pd.DataFrame(metric_rows)
qual_summary.to_csv(QUAL_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print("\n=== Average judge scores ===")
print(qual_summary.to_string(index=False, formatters={
    "E1_single": "{:.3f}".format,
    "E4_ClawTeam": "{:.3f}".format,
    "delta_E4_minus_E1": "{:+.3f}".format,
}))

print("\nSaved summary:", QUAL_SUMMARY_CSV)
qual_summary

=== TumorBoard qualitative per-case summary ===
case_id      winner  e1_avg_score  e4_avg_score  e1_medical_accuracy  e4_medical_accuracy  e1_plan_completeness  e4_plan_completeness  e1_mdt_reasoning  e4_mdt_reasoning  e1_actionable_timeline  e4_actionable_timeline
 tb_008 E4_ClawTeam          4.25          5.00                  4.0                  5.0                   5.0                   5.0               3.0               5.0                     5.0                     5.0
 tb_025 E4_ClawTeam          3.25          5.00                  4.0                  5.0                   4.0                   5.0               2.0               5.0                     3.0                     5.0
 tb_007   E1_single          4.75          4.25                  5.0                  4.0                   5.0                   4.0               4.0               5.0                     5.0                     4.0
 tb_029 E4_ClawTeam          2.25          5.00                  2.0            

,metric,E1_single,E4_ClawTeam,delta_E4_minus_E1
0,medical_accuracy,4.0,4.80,0.80
1,plan_completeness,4.4,4.80,0.40
2,mdt_reasoning,2.6,5.00,2.40
3,actionable_timeline,4.2,4.80,0.60
4,avg_score,3.8,4.85,1.05


## 写报告时的推荐表述

> We further conduct a small qualitative TumorBoard evaluation on 5 free-form oncology cases.
> A strong single-agent baseline (E1) and the full ClawTeam system (E4) are judged by an LLM evaluator
> across medical accuracy, plan completeness, MDT reasoning, and actionable timeline.
> This auxiliary evaluation is not a medical gold standard, but it directly probes the harness value that
> closed-form MCQ accuracy cannot capture.